# UnseenNAS — Full Pipeline on Google Colab (TPU v4 / GPU / CPU)

This notebook runs the **complete UnseenNAS competition pipeline** end-to-end:

```
DataProcessor  →  NAS (AZ-NAS zero-cost proxy + Aging Evolution)  →  Trainer  →  Predict  →  Score
```

### Before running

1. **Select hardware** → Runtime → Change runtime type → **TPU v4** (or T4 / V100 / A100 GPU)
2. **Prepare datasets** → Upload to Google Drive *or* let Section 1 download two small ones automatically
3. **Run all cells** → Runtime → Run all

### Expected duration per dataset

| Hardware | NAS (150 rounds) | Training epoch | Full pipeline |
|---|---|---|---|
| TPU v4   | ~45 s | ~20–60 s | **15–30 min** |
| A100 GPU | ~30 s | ~10–30 s | **10–20 min** |
| T4 GPU   | ~90 s | ~30–90 s | **20–40 min** |
| CPU only | ~25 min | ~5–15 min | **several hours** |

> If running on CPU, reduce `NAS_N_ROUNDS` and `TIME_LIMIT_HOURS` in Section 2.


## 0 — Environment Setup

In [ ]:
import subprocess, sys, os, json, time, math, copy, pickle, traceback, warnings
warnings.filterwarnings('ignore')
import numpy as np

print("=" * 60)
print("  Detecting hardware...")
print("=" * 60)

import torch

# ── Try TPU (torch_xla) first ─────────────────────────────────────────────────
try:
    import torch_xla
    import torch_xla.core.xla_model as xm
    DEVICE = xm.xla_device()
    XM     = xm
    ACCEL  = "TPU"
    print(f"✓  TPU  |  device={DEVICE}  |  torch_xla {torch_xla.__version__}")
except Exception as _tpu_err:
    XM = None
    if torch.cuda.is_available():
        DEVICE = torch.device('cuda')
        ACCEL  = "GPU"
        props  = torch.cuda.get_device_properties(0)
        print(f"✓  GPU  |  {props.name}  |  {props.total_memory/1e9:.1f} GB VRAM")
    else:
        DEVICE = torch.device('cpu')
        ACCEL  = "CPU"
        import multiprocessing
        print(f"⚠  CPU  |  {multiprocessing.cpu_count()} cores  (slow — consider a GPU/TPU runtime)")

print(f"   PyTorch {torch.__version__}  |  Python {sys.version.split()[0]}")


In [ ]:
# Install missing Python dependencies (scikit-learn etc.)
_reqs = ['scikit-learn', 'numpy', 'scipy', 'matplotlib']
subprocess.run([sys.executable, '-m', 'pip', 'install'] + _reqs + ['-q'], check=True)
print("✓  Python dependencies ready")


In [ ]:
# Clone or update the UnseenNAS repository
REPO_URL = "https://github.com/jesusllg/unseennas-automl-competition"
REPO_DIR = "/content/UnseenNAS"

if os.path.exists(f"{REPO_DIR}/.git"):
    r = subprocess.run(
        ['git', '-C', REPO_DIR, 'pull', '--ff-only'],
        capture_output=True, text=True
    )
    print("Repo updated:", r.stdout.strip() or "already up to date")
else:
    subprocess.run(['git', 'clone', '--depth=1', REPO_URL, REPO_DIR], check=True)
    print("✓  Repo cloned")

sys.path.insert(0, f"{REPO_DIR}/submission")
os.chdir(REPO_DIR)
print(f"   Working dir: {os.getcwd()}")
print(f"   Submission path added: {REPO_DIR}/submission")


## 1 — Datasets

**Option A — Google Drive (recommended)**
1. Create the folder `My Drive / UnseenNAS / datasets /` in your Google Drive
2. Inside it, put one subfolder per dataset (e.g. `AddNIST/`, `CIFARTile/`)
3. Each subfolder needs: `train_x.npy`, `train_y.npy`, `valid_x.npy`, `valid_y.npy`, `test_x.npy`, `metadata`

**Option B — Auto-download**
If no Drive datasets are found, the next cell will download **AddNIST** and **CIFARTile** (~200 MB total) automatically.


In [ ]:
# Mount Google Drive and locate datasets
DATASETS_DIR = None

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    print("Google Drive mounted")

    _candidates = [
        '/content/drive/MyDrive/UnseenNAS/datasets',
        '/content/drive/My Drive/UnseenNAS/datasets',
        '/content/drive/MyDrive/datasets',
        '/content/drive/My Drive/datasets',
    ]
    for _p in _candidates:
        if os.path.isdir(_p):
            _subdirs = [d for d in os.listdir(_p) if os.path.isdir(os.path.join(_p, d))]
            if _subdirs:
                DATASETS_DIR = _p
                print(f"✓  Datasets found: {DATASETS_DIR}")
                print(f"   Folders: {_subdirs}")
                break
    if DATASETS_DIR is None:
        print("Drive mounted but no dataset folders found. Expected one of:")
        for _p in _candidates:
            print(f"   {_p}")
except Exception as _e:
    print(f"Google Drive not available: {_e}")


In [ ]:
# Auto-download if no Drive datasets found
# (Skips automatically if DATASETS_DIR is already set from Drive)

if DATASETS_DIR is None:
    print("No Drive datasets — downloading AddNIST + CIFARTile (~200 MB)…")
    DATASETS_DIR = '/content/datasets'
    os.makedirs(DATASETS_DIR, exist_ok=True)
    subprocess.run(
        [sys.executable, f'{REPO_DIR}/download_datasets.py',
         'AddNIST', 'CIFARTile', '--out', DATASETS_DIR],
        cwd=REPO_DIR, check=False
    )
else:
    print("Skipping download — using Drive datasets")

# List available datasets
assert DATASETS_DIR is not None, "DATASETS_DIR is not set"
available_datasets = sorted([
    d for d in os.listdir(DATASETS_DIR)
    if os.path.isdir(os.path.join(DATASETS_DIR, d))
    and os.path.exists(os.path.join(DATASETS_DIR, d, 'metadata'))
])

print(f"\nAvailable ({len(available_datasets)} datasets):")
print(f"{'Name':15s}  {'Shape':25s}  {'Classes':>7s}  {'Benchmark':>10s}")
print("-" * 65)
for _d in available_datasets:
    with open(os.path.join(DATASETS_DIR, _d, 'metadata')) as _f:
        _m = json.load(_f)
    _shape = str(_m['input_shape'])
    print(f"{_d:15s}  {_shape:25s}  {_m['num_classes']:>7d}  "
          f"{str(_m.get('benchmark','?')):>10s}")


## 2 — Configuration

Edit the values in the next cell, then run it.
All hyperparameters are documented in the [README](https://github.com/jesusllg/unseennas-automl-competition/blob/main/README.md#8-complete-hyperparameter-reference).


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#   CONFIGURATION — edit these values
# ══════════════════════════════════════════════════════════════════════════

# Which datasets to run  (list of names shown above)
# Examples:
#   SELECTED = available_datasets          # run ALL
#   SELECTED = ['AddNIST', 'CIFARTile']   # run two specific ones
#   SELECTED = available_datasets[:1]      # run only the first one
SELECTED = available_datasets[:1]

# Total time budget PER dataset (in hours)
#   0.25 = 15 min   |   0.5 = 30 min   |   1.0 = 1 hour   |   2.0 = 2 hours
TIME_LIMIT_HOURS = 1.0

# ── NAS hyperparameters ──────────────────────────────────────────────────────
# Set to None to use automatic values based on dataset benchmark difficulty.
# Refer to README Section 8.1 for details.
NAS_N_POPULATION = None    # Population size         (auto: 20 – 40)
NAS_N_ROUNDS     = None    # Evolution rounds        (auto: 60 – 150)
NAS_TOURNAMENT_K = None    # Tournament size         (auto: 5 – 10)
NAS_SEARCH_FRAC  = None    # Fraction of time for search (auto: 0.20 – 0.35)

# ── AZ-NAS proxy ─────────────────────────────────────────────────────────────
# Weight on complexity penalty — higher → system prefers smaller models
AZ_NAS_LAMBDA_C = 0.1     # default 0.1; try 0.0 (ignore size) or 0.3 (small models)

# ── Save to Drive ─────────────────────────────────────────────────────────────
# After the run, copy predictions to Drive automatically?
SAVE_TO_DRIVE = True   # set False if you don't want to copy back

# ══════════════════════════════════════════════════════════════════════════

print("Configuration:")
print(f"  Datasets         : {SELECTED}")
print(f"  Time per dataset : {TIME_LIMIT_HOURS} h  ({TIME_LIMIT_HOURS*60:.0f} min)")
print(f"  Device           : {ACCEL}  ({DEVICE})")
print(f"  NAS pop/rounds/k : {NAS_N_POPULATION}/{NAS_N_ROUNDS}/{NAS_TOURNAMENT_K}  (None = auto)")
print(f"  AZ-NAS lambda_c  : {AZ_NAS_LAMBDA_C}")
print(f"  Save to Drive    : {SAVE_TO_DRIVE}")


## 3 — Run the Full Pipeline

> **This is the main cell.** Run it and wait — progress will be printed live.
> On TPU v4 with `TIME_LIMIT_HOURS = 1.0` expect **~15–30 min per dataset**.
>
> The pipeline for each dataset:
> 1. **DataProcessor** — normalise, augment, build DataLoaders
> 2. **NAS** — Aging Evolution + AZ-NAS zero-cost proxy (no training during search)
> 3. **Trainer** — AdamW + CosineAnnealingLR + AMP until time runs out
> 4. **Predict** — run best model on test split
> 5. **Score** — compare vs test labels (if available)


In [ ]:
# ── Helpers ──────────────────────────────────────────────────────────────────

class Clock:
    """Countdown clock compatible with the submission Clock API."""
    def __init__(self, hours):
        self.limit = time.perf_counter() + hours * 3600
    def check(self):
        return self.limit - time.perf_counter()

def _show_time(s):
    s = max(0, s)
    if s < 60:   return f"{s:.1f}s"
    if s < 3600: m, s = int(s // 60), int(s % 60);  return f"{m}m,{s}s"
    h = int(s // 3600); m = int((s % 3600) // 60); return f"{h}h,{m}m"

# ── Apply NAS config overrides (monkey-patch _search_params) ─────────────────
import nas as _nas_mod

if any(v is not None for v in [NAS_N_POPULATION, NAS_N_ROUNDS, NAS_TOURNAMENT_K, NAS_SEARCH_FRAC]):
    _orig_sp = _nas_mod._search_params
    def _patched_sp(benchmark):
        sf, np_, nr, tk = _orig_sp(benchmark)
        if NAS_SEARCH_FRAC   is not None: sf  = NAS_SEARCH_FRAC
        if NAS_N_POPULATION  is not None: np_ = NAS_N_POPULATION
        if NAS_N_ROUNDS      is not None: nr  = NAS_N_ROUNDS
        if NAS_TOURNAMENT_K  is not None: tk  = NAS_TOURNAMENT_K
        return sf, np_, nr, tk
    _nas_mod._search_params = _patched_sp
    print("✓  NAS overrides applied")

# ── Apply AZ-NAS lambda_c override ───────────────────────────────────────────
if AZ_NAS_LAMBDA_C != 0.1:
    from search_space import proxies as _px
    import search_space as _ss
    _orig_az = _px.az_nas_score
    def _patched_az(model, batch_x, device, lambda_c=AZ_NAS_LAMBDA_C, reinit=True):
        return _orig_az(model, batch_x, device, lambda_c=lambda_c, reinit=reinit)
    _px.az_nas_score = _patched_az
    _ss.az_nas_score = _patched_az
    print(f"✓  AZ-NAS lambda_c = {AZ_NAS_LAMBDA_C}")

# ── Link datasets dir into repo root (main.py expects datasets/ here) ─────────
_ds_link = f"{REPO_DIR}/datasets"
if os.path.islink(_ds_link):
    os.unlink(_ds_link)
if not os.path.exists(_ds_link):
    os.symlink(DATASETS_DIR, _ds_link)

os.makedirs(f"{REPO_DIR}/predictions", exist_ok=True)

# ── Import submission modules ─────────────────────────────────────────────────
from data_processor import DataProcessor
from nas import NAS
from trainer import Trainer

print("\n" + "=" * 65)
print(f"  Starting pipeline on {len(SELECTED)} dataset(s)  |  {ACCEL}")
print("=" * 65)

all_results = []

for ds_name in SELECTED:
    ds_path = os.path.join(DATASETS_DIR, ds_name)

    print(f"\n{'='*65}")
    print(f"  Dataset: {ds_name}")
    print(f"{'='*65}")

    with open(os.path.join(ds_path, 'metadata')) as _f:
        metadata = json.load(_f)

    codename = metadata['codename']
    clock    = Clock(TIME_LIMIT_HOURS)
    t_start  = time.perf_counter()

    def _load(split, kind):
        return np.load(os.path.join(ds_path, f'{split}_{kind}.npy'))

    train_x, train_y = _load('train', 'x'), _load('train', 'y')
    valid_x, valid_y = _load('valid', 'x'), _load('valid', 'y')
    test_x           = _load('test',  'x')

    print(f"  train: {train_x.shape}  valid: {valid_x.shape}  test: {test_x.shape}")
    print(f"  classes={metadata['num_classes']}  benchmark={metadata.get('benchmark','?')}")
    print(f"  time budget: {TIME_LIMIT_HOURS}h  |  remaining: {_show_time(clock.check())}")

    try:
        # 1. DataProcessor
        print("\n── DataProcessor ─────────────────────────────────────────────")
        dp = DataProcessor(train_x, train_y, valid_x, valid_y, test_x, metadata, clock)
        train_dl, valid_dl, test_dl = dp.process()
        print(f"  batch_size={metadata.get('batch_size', '?')}")

        # 2. NAS
        print("\n── NAS ───────────────────────────────────────────────────────")
        print(f"  Remaining: {_show_time(clock.check())}")
        model = NAS(train_dl, valid_dl, metadata, clock).search()
        n_params = sum(p.numel() for p in model.parameters())
        print(f"  Selected model: {n_params:,} parameters")

        # 3. Trainer (pass the Colab device — GPU or TPU)
        print("\n── Trainer ───────────────────────────────────────────────────")
        print(f"  Remaining: {_show_time(clock.check())}  |  device: {DEVICE}")
        trainer = Trainer(model, DEVICE, train_dl, valid_dl, metadata, clock)
        model = trainer.train()

        # 4. Predict
        print("\n── Predicting ────────────────────────────────────────────────")
        preds = trainer.predict(test_dl)
        preds = np.array(preds)
        print(f"  {len(preds)} predictions  |  classes: {np.unique(preds).tolist()}")

        # 5. Save predictions
        elapsed = time.perf_counter() - t_start
        pred_path  = f'{REPO_DIR}/predictions/{codename}.npy'
        stats_path = f'{REPO_DIR}/predictions/{codename}_stats.pkl'
        np.save(pred_path, preds)
        with open(stats_path, 'wb') as _f:
            pickle.dump({'Failed': False, 'Runtime': elapsed, 'Params': n_params}, _f)

        # 6. Score against test labels (if available)
        label_path = os.path.join(ds_path, 'test_y.npy')
        acc = None
        if os.path.exists(label_path):
            labels = np.load(label_path)
            acc = float((preds == labels).mean()) * 100
            bm  = float(metadata.get('benchmark', 0))
            delta = acc - bm
            print(f"\n  ┌─ RESULT ─────────────────────────────────┐")
            print(f"  │  Accuracy : {acc:6.2f}%                        │")
            print(f"  │  Benchmark: {bm:6.2f}%   Δ = {delta:+.2f}%          │")
            print(f"  │  Time     : {_show_time(elapsed):10s}  Params: {n_params/1e6:.2f}M │")
            print(f"  └──────────────────────────────────────────────┘")
        else:
            print(f"\n  ✓ Predictions saved (no test labels for scoring)")

        all_results.append({
            'dataset': ds_name, 'codename': codename,
            'acc': acc, 'params': n_params,
            'time_s': elapsed, 'status': 'OK',
        })

    except Exception as _exc:
        elapsed = time.perf_counter() - t_start
        print(f"\n  ✗ FAILED: {_exc}")
        traceback.print_exc()
        stats_path = f'{REPO_DIR}/predictions/{codename}_stats.pkl'
        with open(stats_path, 'wb') as _f:
            pickle.dump({'Failed': True, 'Runtime': -1, 'Params': None}, _f)
        all_results.append({
            'dataset': ds_name, 'status': 'FAILED', 'error': str(_exc),
        })

print("\n" + "=" * 65)
print("  Pipeline complete!")
print("=" * 65)


## 4 — Results

In [ ]:
# Results table
print(f"{'Dataset':15s}  {'Status':6s}  {'Accuracy':>10s}  {'Benchmark':>10s}  "
      f"{'Delta':>8s}  {'Params':>8s}  {'Time':>8s}")
print("─" * 80)

for r in all_results:
    ds   = r['dataset']
    stat = r['status']

    if stat != 'OK':
        print(f"{ds:15s}  {stat:6s}  {r.get('error','')[:40]}")
        continue

    with open(os.path.join(DATASETS_DIR, ds, 'metadata')) as _f:
        _m = json.load(_f)

    acc = r.get('acc')
    bm  = float(_m.get('benchmark', 0))
    par = r.get('params', 0) or 0
    t   = r.get('time_s', 0)

    acc_s  = f"{acc:.2f}%" if acc is not None else "N/A"
    bm_s   = f"{bm:.2f}%"
    delta  = f"{acc - bm:+.2f}%" if acc is not None else "N/A"
    par_s  = f"{par/1e6:.2f}M" if par else "N/A"

    print(f"{ds:15s}  {'OK':6s}  {acc_s:>10s}  {bm_s:>10s}  "
          f"{delta:>8s}  {par_s:>8s}  {_show_time(t):>8s}")


In [ ]:
# Accuracy vs Benchmark bar chart
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

ok = [r for r in all_results if r.get('status') == 'OK' and r.get('acc') is not None]

if ok:
    fig, ax = plt.subplots(figsize=(max(6, len(ok) * 2), 5))
    names = [r['dataset'] for r in ok]
    accs  = [r['acc'] for r in ok]

    bms = []
    for r in ok:
        with open(os.path.join(DATASETS_DIR, r['dataset'], 'metadata')) as _f:
            _m = json.load(_f)
        bms.append(float(_m.get('benchmark', 0)))

    x = np.arange(len(names))
    w = 0.35
    ax.bar(x - w/2, bms,  w, label='Baseline benchmark', color='#94a3b8', alpha=0.9)
    bars = ax.bar(x + w/2, accs, w, label='Our NAS result',   color='#6366f1', alpha=0.95)

    for bar, val in zip(bars, accs):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.4,
                f'{val:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')

    ax.set_xticks(x)
    ax.set_xticklabels(names, rotation=15, ha='right', fontsize=11)
    ax.set_ylabel('Accuracy (%)', fontsize=12)
    ax.set_title(f'UnseenNAS Results  |  {ACCEL}', fontsize=14, fontweight='bold')
    ax.legend(fontsize=11)
    ax.set_ylim(0, max(max(accs), max(bms)) * 1.18)
    ax.spines[['top','right']].set_visible(False)
    ax.grid(axis='y', alpha=0.3)

    out_chart = f'{REPO_DIR}/predictions/results_chart.png'
    plt.tight_layout()
    plt.savefig(out_chart, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Chart saved → {out_chart}")
else:
    print("No successful results to plot yet")


In [ ]:
# Save predictions folder to Google Drive (optional)
if SAVE_TO_DRIVE:
    try:
        import shutil
        drive_out = '/content/drive/MyDrive/UnseenNAS/predictions'
        shutil.copytree(
            f'{REPO_DIR}/predictions', drive_out,
            dirs_exist_ok=True
        )
        print(f"✓  Predictions saved to Drive: {drive_out}")
    except Exception as _e:
        print(f"Could not save to Drive: {_e}")
        print(f"Predictions are in: {REPO_DIR}/predictions/")
else:
    print(f"Predictions are in: {REPO_DIR}/predictions/")
    print("(Set SAVE_TO_DRIVE = True in Section 2 to copy to Drive automatically)")


## Tips & Troubleshooting

### Speeding things up

| Goal | What to change |
|---|---|
| Faster search | `NAS_N_ROUNDS = 30`, `NAS_N_POPULATION = 10` in Section 2 |
| Shorter run | `TIME_LIMIT_HOURS = 0.25` (15 min) |
| Run all datasets | `SELECTED = available_datasets` |
| Focus on training | `NAS_SEARCH_FRAC = 0.10` (10% of time for NAS, 90% for training) |
| Prefer small models | `AZ_NAS_LAMBDA_C = 0.3` |

### Common issues

**`FileNotFoundError: predictions/...`** — Fixed in latest repo (`git pull` in Section 0 cell handles this).

**TPU gives lower accuracy than GPU** — Both should be similar; TPU may need more epochs to warm up.
Increase `TIME_LIMIT_HOURS` or decrease `NAS_N_ROUNDS` to leave more time for training.

**`torch_xla` not found on TPU runtime** — It should be pre-installed on Colab TPU v4.
If not, add this cell before the environment setup:
```python
!pip install torch_xla[tpu] -f https://storage.googleapis.com/libtpu-releases/index.html -q
```

**Dataset download failed** — Download manually from the DOI links in the README and upload to Drive.

**OOM (out of memory)** — Add `NAS_N_POPULATION = 20` and the repair system will keep models smaller.

### Questions / bugs
→ [GitHub Issues](https://github.com/jesusllg/unseennas-automl-competition/issues)
